In [ ]:
omics_model_package/
 ├─ omics_mlp.pt         # 학습된 모델 (이미 있음)
 ├─ scaler.pkl           # 저장 완료
 ├─ pca.pkl              # 저장 완료
 ├─ omics_model.py       # 위 MLP 클래스 코드
 └─ predict_omics.py     # 불러오기 + 추론 코드


In [ ]:
# 최종 구조 코드입니다

# omics_model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: int = 256, dropout: float = 0.2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, 1)   # 이진분류 (AD vs Control)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        x = F.relu(self.fc2(x))
        x = self.drop(x)
        x = self.out(x).squeeze(-1)  # 로짓 출력
        return x


In [ ]:
# 추론용 코드입니다.

import numpy as np
import torch
import joblib
import torch.nn.functional as F
from omics_model import MLP

# 전처리기 불러오기
scaler = joblib.load("scaler.pkl")
pca    = joblib.load("pca.pkl")

# 모델 불러오기
checkpoint = torch.load("omics_mlp.pt", map_location="cpu")
in_dim     = checkpoint["in_dim"]
hidden     = checkpoint["hidden"]
dropout    = checkpoint["dropout"]
T          = checkpoint.get("temperature", 1.0)  # 온도 보정 값

model = MLP(in_dim=in_dim, hidden=hidden, dropout=dropout)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

def predict_omics(raw_data: np.ndarray):
    """
    raw_data : (n_samples, n_features) numpy array (비전처리 원본)
    return   : 확률 (AD 위험도)
    """
    # 1) NaN/Inf 처리
    raw_data = np.nan_to_num(raw_data, nan=0.0, posinf=0.0, neginf=0.0)

    # 2) Scaling
    X_scaled = scaler.transform(raw_data)

    # 3) PCA
    X_proc = pca.transform(X_scaled)

    # 4) Tensor 변환
    X_tensor = torch.tensor(X_proc, dtype=torch.float32)

    # 5) 모델 추론 + 온도 보정
    with torch.no_grad():
        logits = model(X_tensor)
        logits = logits / max(T, 1e-3)   # Temperature scaling
        probs = torch.sigmoid(logits).numpy()

    return probs
